In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

class YouTubeFactorAnalysis:
    """
    Исследовательский пайплайн для декомпозиции признаков YouTube-каналов
    с использованием метода главных компонент (PCA).
    """
    def __init__(self, n_components: int = 2):
        self.n_components = n_components
        self.scaler = StandardScaler()
        self.pca = PCA(n_components=n_components)

        # Целевые числовые признаки для сжатия
        self.feature_cols = [
            'subscribers', 'video views', 'highest_monthly_earnings',
            'lowest_monthly_earnings', 'video_views_for_the_last_30_days'
        ]

    def analyze(self, data_path: str):
        if not os.path.exists(data_path):
            print(f"[ERROR] Файл {data_path} не найден.")
            return

        # 1. Загрузка и очистка от зануленных/пропущенных данных
        df = pd.read_csv(data_path, encoding='ISO-8859-1')

        # Дропаем строки, где важные метрики равны нулю или NaN
        df = df.dropna(subset=self.feature_cols + ['category'])
        df = df[(df[self.feature_cols] > 0).all(axis=1)]

        X = df[self.feature_cols].values
        labels = df['category'].values

        print(f"[INFO] Размерность очищенной матрицы признаков: {X.shape}")

        # 2. Масштабирование признаков (PCA критически чувствителен к разным масштабам!)
        X_scaled = self.scaler.fit_transform(X)

        # 3. Применение PCA
        X_pca = self.pca.fit_transform(X_scaled)

        # 4. Анализ объясненной дисперсии (Explained Variance)
        explained_variance = self.pca.explained_variance_ratio_
        print("\n=== Анализ главных компонент (PCA) ===")
        for i, var in enumerate(explained_variance):
            print(f"Главная компонента PC{i+1} объясняет: {var*100:.2f}% дисперсии данных")
        print(f"Суммарная объясненная дисперсия ({self.n_components}D): {sum(explained_variance)*100:.2f}%")

        # 5. Анализ нагрузок (Интерпретация компонент)
        # Показывает, вклад каких исходных фич максимален в каждую компоненту
        loadings = pd.DataFrame(
            self.pca.components_.T,
            columns=[f'PC{i+1}' for i in range(self.n_components)],
            index=self.feature_cols
        )
        print("\n=== Матрица весовых нагрузок признаков (Loadings) ===")
        print(loadings)

        # 6. Экспорт результатов в структурированный датафрейм
        result_df = pd.DataFrame(X_pca, columns=[f'PC{i+1}' for i in range(self.n_components)])
        result_df['category'] = labels
        result_df.to_csv("youtube_pca_factors.csv", index=False)
        print("\n[INFO] Результаты факторного анализа сохранены в 'youtube_pca_factors.csv'")

if __name__ == "__main__":
    # Для работы требуется оригинальный датасет Global YouTube Statistics.csv
    analyzer = YouTubeFactorAnalysis(n_components=2)
    analyzer.analyze("Global YouTube Statistics.csv")

[INFO] Размерность очищенной матрицы признаков: (842, 5)

=== Анализ главных компонент (PCA) ===
Главная компонента PC1 объясняет: 78.36% дисперсии данных
Главная компонента PC2 объясняет: 18.59% дисперсии данных
Суммарная объясненная дисперсия (2D): 96.95%

=== Матрица весовых нагрузок признаков (Loadings) ===
                                       PC1       PC2
subscribers                       0.368799  0.656995
video views                       0.411763  0.518722
highest_monthly_earnings          0.481060 -0.316196
lowest_monthly_earnings           0.481156 -0.315671
video_views_for_the_last_30_days  0.481154 -0.315685

[INFO] Результаты факторного анализа сохранены в 'youtube_pca_factors.csv'
